# Ablation Study — Multimodal Fusion Architecture

Evaluates the contribution of each component of the
**ImageShapeFusionTransformer** by training 5 model variants under identical
conditions (biological-only label set, same splits, same training budget):

| Variant | Image branch | Shape branch | Fusion mechanism |
|---------|-------------|--------------|-------------------|
| **Full Model** | ConvNeXt-Tiny | 9 shape tokens | Transformer encoder |
| **Image-Only** | ConvNeXt-Tiny | — | — (image features only) |
| **Shape-Only** | — | 9 shape features (MLP) | — |
| **No-Fusion** | ConvNeXt-Tiny | MLP on shape feats | Concatenation (no cross-attention) |
| **No-Transformer** | ConvNeXt-Tiny | 9 shape tokens | Mean-pooling (no Transformer encoder) |

**Labels (biological-only, Table 10):** cell_line, culture_medium,
seeding_density, timepoint, formation_method

**Expected ordering (paper):**
```
Full Model (0.8681) > Image-Only (0.8662) > No-Transformer (0.8650)
  > No-Fusion (0.8610) >> Shape-Only (0.5530)
```

In [ ]:
# !pip install timm --quiet

In [ ]:
import os, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import tifffile
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Config
class CFG:
    metadata_csv = "../data/slimia_metadata.csv"
    shape_csv    = "../data/shape_features_with_metadata.csv"
    ckpt_dir     = "../checkpoints/ipp/ablation/"
    output_dir   = "../results/ablations/"

    # Biological-only labels (Table 10)
    label_columns = [
        "cell_line", "culture_medium", "seeding_density",
        "timepoint", "formation_method"
    ]

    shape_features = [
        "area", "perimeter", "eccentricity", "solidity", "extent",
        "equivalent_diameter", "major_axis_length", "minor_axis_length", "circularity"
    ]

    train_reps = ["T1", "T2", "T3", "T4"]
    val_reps   = ["T5", "T8"]
    test_reps  = ["T6", "T7"] + [f"T{i}" for i in range(9, 25)]

    d_model  = 256; n_heads = 4; n_layers = 3; ff_dim = 512; dropout = 0.1
    image_size  = 224
    batch_size  = 32
    num_epochs  = 100
    patience    = 20
    lr          = 1e-4
    weight_decay= 1e-2
    num_workers = 2
    seed        = 42
    device      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.ckpt_dir,   exist_ok=True)
os.makedirs(CFG.output_dir, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(CFG.seed)

## Data Loading

In [ ]:
# Merge metadata + shape features
meta_df  = pd.read_csv(CFG.metadata_csv)
shape_df = pd.read_csv(CFG.shape_csv)
meta_df["full_path"]  = meta_df["full_path"].astype(str).str.strip()
meta_df["fname"]      = meta_df["full_path"].apply(lambda x: Path(x).name)
shape_df["fname"]     = shape_df["image_path"].apply(lambda x: Path(x).name)

merged = pd.merge(meta_df, shape_df, on="fname", how="inner", suffixes=("", "_shp"))

label_encoders, label_dims = {}, {}
for col in CFG.label_columns:
    le = LabelEncoder()
    merged[col] = merged[col].astype(str)
    merged[col + "_enc"] = le.fit_transform(merged[col])
    label_encoders[col]  = le
    label_dims[col]      = merged[col + "_enc"].nunique()

for sf in CFG.shape_features:
    merged[sf] = pd.to_numeric(merged[sf], errors="coerce").fillna(0.0)

train_df = merged[merged["technical_rep"].isin(CFG.train_reps)].copy()
val_df   = merged[merged["technical_rep"].isin(CFG.val_reps)].copy()
test_df  = merged[merged["technical_rep"].isin(CFG.test_reps)].copy()

shape_mean = train_df[CFG.shape_features].mean().values.astype(np.float32)
shape_std  = train_df[CFG.shape_features].std().replace(0, 1).values.astype(np.float32)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Label dims:", label_dims)

# Class weights
class_weights = {}
for col in CFG.label_columns:
    y  = train_df[col + "_enc"].values
    nc = label_dims[col]
    counts = np.bincount(y, minlength=nc).astype(np.float32)
    w = np.where(counts > 0, len(y) / (nc * np.maximum(counts, 1)), 0.0)
    class_weights[col] = torch.tensor(w, dtype=torch.float32).to(CFG.device)

In [ ]:
# Dataset
def load_image_rgb(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        arr = tifffile.imread(path)
        if arr.ndim == 2: arr = np.stack([arr]*3, -1)
        arr = arr.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        return Image.fromarray((arr*255).astype(np.uint8)).convert("RGB")


class AblationDataset(Dataset):
    """Returns (image, shape_vec, labels) — image and shape are always provided;
    individual model variants ignore whichever branch they don't use."""
    def __init__(self, frame, transform=None):
        self.df        = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = load_image_rgb(row["full_path"])
        if self.transform: img = self.transform(img)
        shape_vec = torch.tensor(row[CFG.shape_features].to_numpy(dtype=np.float32))
        labels    = torch.tensor([int(row[c + "_enc"]) for c in CFG.label_columns],
                                  dtype=torch.long)
        return img, shape_vec, labels


train_tf = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.RandomHorizontalFlip(), T.RandomRotation(10),
    T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
val_tf = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])

train_loader = DataLoader(AblationDataset(train_df, train_tf),
    batch_size=CFG.batch_size, shuffle=True,  num_workers=CFG.num_workers, pin_memory=True)
val_loader   = DataLoader(AblationDataset(val_df, val_tf),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
test_loader  = DataLoader(AblationDataset(test_df, val_tf),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

## Model Variants

In [ ]:
# Shared shape-token projection (used by Full / No-Transformer)
class ShapeTokenizer(nn.Module):
    """Projects each of the 9 normalised shape features into a D-dim token."""
    def __init__(self, n_features, D, shape_mean, shape_std):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(1, D) for _ in range(n_features)])
        self.n    = n_features
        self.register_buffer("mean", torch.tensor(shape_mean))
        self.register_buffer("std",  torch.tensor(shape_std))

    def forward(self, shape_feats):
        x = (shape_feats - self.mean) / (self.std + 1e-6)
        return torch.cat([self.proj[i](x[:, i:i+1]).unsqueeze(1)
                          for i in range(self.n)], dim=1)   # (B, n, D)


def make_heads(D, label_dims):
    return nn.ModuleDict({
        lab: nn.Sequential(
            nn.LayerNorm(D), nn.Linear(D, D), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(D, ncls))
        for lab, ncls in label_dims.items()
    })

In [ ]:
# (1) Full Model: Image + Shape + Transformer Fusion
class FullFusionModel(nn.Module):
    """Identical to ImageShapeFusionTransformer (Table 2 in the paper)."""
    def __init__(self, label_dims, shape_mean, shape_std):
        super().__init__()
        D = CFG.d_model
        self.backbone   = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.image_proj = nn.Linear(self.backbone.num_features, D)
        self.shape_tok  = ShapeTokenizer(len(CFG.shape_features), D, shape_mean, shape_std)
        n_shape = len(CFG.shape_features)
        self.pos_embed  = nn.Parameter(torch.randn(1, 1 + n_shape, D) * 0.02)
        enc = nn.TransformerEncoderLayer(D, CFG.n_heads, CFG.ff_dim,
                                         CFG.dropout, activation="gelu")
        self.transformer = nn.TransformerEncoder(enc, CFG.n_layers)
        self.norm  = nn.LayerNorm(D)
        self.heads = make_heads(D, label_dims)

    def forward(self, images, shape_feats):
        img_tok = self.image_proj(self.backbone(images)).unsqueeze(1)
        s_toks  = self.shape_tok(shape_feats)
        seq     = torch.cat([img_tok, s_toks], dim=1) + self.pos_embed
        fused   = self.norm(self.transformer(seq.permute(1,0,2))[0])
        return {lab: head(fused) for lab, head in self.heads.items()}

In [ ]:
# (2) Image-Only: ConvNeXt-Tiny, no shape features
class ImageOnlyModel(nn.Module):
    def __init__(self, label_dims, shape_mean=None, shape_std=None):
        super().__init__()
        D = CFG.d_model
        self.backbone   = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.image_proj = nn.Linear(self.backbone.num_features, D)
        self.heads = make_heads(D, label_dims)

    def forward(self, images, shape_feats):
        feat = self.image_proj(self.backbone(images))
        return {lab: head(feat) for lab, head in self.heads.items()}

In [ ]:
# (3) Shape-Only: MLP on the 9 morphometric features only
class ShapeOnlyModel(nn.Module):
    def __init__(self, label_dims, shape_mean, shape_std):
        super().__init__()
        D = CFG.d_model
        n_shape = len(CFG.shape_features)
        self.register_buffer("mean", torch.tensor(shape_mean))
        self.register_buffer("std",  torch.tensor(shape_std))
        self.mlp = nn.Sequential(
            nn.Linear(n_shape, D), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(D, D), nn.GELU())
        self.heads = make_heads(D, label_dims)

    def forward(self, images, shape_feats):
        x    = (shape_feats - self.mean) / (self.std + 1e-6)
        feat = self.mlp(x)
        return {lab: head(feat) for lab, head in self.heads.items()}

In [ ]:
# (4) No-Fusion: separate image/shape branches, concatenated (no cross-attn)
class NoFusionModel(nn.Module):
    """
    Image and shape pathways are present but NOT interactively fused —
    each branch is processed independently and outputs are concatenated
    before the classification heads (no Transformer cross-attention).
    """
    def __init__(self, label_dims, shape_mean, shape_std):
        super().__init__()
        D = CFG.d_model
        n_shape = len(CFG.shape_features)
        self.backbone   = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.image_proj = nn.Linear(self.backbone.num_features, D)
        self.register_buffer("mean", torch.tensor(shape_mean))
        self.register_buffer("std",  torch.tensor(shape_std))
        self.shape_mlp  = nn.Sequential(
            nn.Linear(n_shape, D), nn.GELU(), nn.Dropout(0.1), nn.Linear(D, D))
        # heads take concatenated [image_feat || shape_feat] → 2D
        self.heads = nn.ModuleDict({
            lab: nn.Sequential(
                nn.LayerNorm(2*D), nn.Linear(2*D, D), nn.GELU(),
                nn.Dropout(0.2), nn.Linear(D, ncls))
            for lab, ncls in label_dims.items()
        })

    def forward(self, images, shape_feats):
        img_feat   = self.image_proj(self.backbone(images))
        shape_n    = (shape_feats - self.mean) / (self.std + 1e-6)
        shape_feat = self.shape_mlp(shape_n)
        fused      = torch.cat([img_feat, shape_feat], dim=1)
        return {lab: head(fused) for lab, head in self.heads.items()}

In [ ]:
# (5) No-Transformer: both modalities, fused via mean-pooling (no encoder) 
class NoTransformerModel(nn.Module):
    """
    Retains both modalities (image token + 9 shape tokens) but replaces the
    Transformer fusion encoder with simple mean-pooling across all tokens.
    """
    def __init__(self, label_dims, shape_mean, shape_std):
        super().__init__()
        D = CFG.d_model
        self.backbone   = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.image_proj = nn.Linear(self.backbone.num_features, D)
        self.shape_tok  = ShapeTokenizer(len(CFG.shape_features), D, shape_mean, shape_std)
        self.norm  = nn.LayerNorm(D)
        self.heads = make_heads(D, label_dims)

    def forward(self, images, shape_feats):
        img_tok = self.image_proj(self.backbone(images)).unsqueeze(1)  # (B,1,D)
        s_toks  = self.shape_tok(shape_feats)                           # (B,9,D)
        seq     = torch.cat([img_tok, s_toks], dim=1)                   # (B,10,D)
        fused   = self.norm(seq.mean(dim=1))                            # mean pool
        return {lab: head(fused) for lab, head in self.heads.items()}

## Training & Evaluation

In [ ]:
def compute_metrics(targets, preds):
    per_label = {}
    for col in CFG.label_columns:
        t, p = np.array(targets[col]), np.array(preds[col])
        per_label[col] = dict(
            acc  = accuracy_score(t, p),
            prec = precision_score(t, p, average="macro", zero_division=0),
            rec  = recall_score(t, p, average="macro", zero_division=0),
            f1   = f1_score(t, p, average="macro", zero_division=0),
        )
    macro = {k: np.mean([per_label[c][k] for c in CFG.label_columns])
             for k in ["acc","prec","rec","f1"]}
    return {"per_label": per_label, "macro": macro}


def run_epoch(model, loader, criterions, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    preds, tgts = defaultdict(list), defaultdict(list)

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, shape, labels in tqdm(loader, leave=False,
                                        desc="train" if is_train else "eval"):
            imgs, shape, labels = (imgs.to(CFG.device), shape.to(CFG.device),
                                   labels.to(CFG.device))
            if is_train:
                optimizer.zero_grad()
                with torch.amp.autocast(device_type=CFG.device):
                    outs = model(imgs, shape)
                    loss = sum(criterions[c](outs[c], labels[:, i])
                               for i, c in enumerate(CFG.label_columns))
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                outs = model(imgs, shape)

            for i, c in enumerate(CFG.label_columns):
                preds[c] += outs[c].argmax(1).detach().cpu().tolist()
                tgts[c]  += labels[:, i].detach().cpu().tolist()

    return compute_metrics(tgts, preds)


def train_variant(name, model, train_loader, val_loader):
    model = model.to(CFG.device)
    criterions = {c: nn.CrossEntropyLoss(weight=class_weights[c])
                  for c in CFG.label_columns}
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr,
                                  weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=5, factor=0.5)
    scaler = torch.amp.GradScaler(enabled=(CFG.device == "cuda"))

    best_f1, patience_cnt = 0.0, 0
    ckpt = os.path.join(CFG.ckpt_dir, f"{name}.pth")

    for epoch in range(1, CFG.num_epochs + 1):
        tr = run_epoch(model, train_loader, criterions, optimizer, scaler)
        va = run_epoch(model, val_loader, criterions)
        scheduler.step(va["macro"]["f1"])
        print(f"  [{name}] Epoch {epoch:03d} | "
              f"Train F1 {tr['macro']['f1']:.4f} | Val F1 {va['macro']['f1']:.4f}")

        if va["macro"]["f1"] > best_f1:
            best_f1 = va["macro"]["f1"]; patience_cnt = 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience_cnt += 1
            if patience_cnt >= CFG.patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(torch.load(ckpt, map_location=CFG.device))
    return model

## Run All 5 Variants

In [ ]:
VARIANT_REGISTRY = [
    ("Full Model (Image + Shape + Transformer Fusion)", FullFusionModel),
    ("Image-Only",      ImageOnlyModel),
    ("No-Transformer",  NoTransformerModel),
    ("No-Fusion",       NoFusionModel),
    ("Shape-Only",      ShapeOnlyModel),
]

ablation_results = {}

for name, model_cls in VARIANT_REGISTRY:
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    set_seed(CFG.seed)
    model = model_cls(label_dims, shape_mean, shape_std)
    model = train_variant(name.split(" ")[0], model, train_loader, val_loader)

    test_m = run_epoch(model, test_loader,
                       {c: nn.CrossEntropyLoss(weight=class_weights[c])
                        for c in CFG.label_columns})
    ablation_results[name] = test_m["macro"]
    print(f"  Test macro: {test_m['macro']}")

    # free GPU memory between variants
    del model
    torch.cuda.empty_cache()

## Results Table (Table 10)

In [ ]:
rows = []
for name, m in ablation_results.items():
    rows.append({
        "Model Variant": name,
        "Accuracy":  f"{m['acc']:.4f}",
        "Precision": f"{m['prec']:.4f}",
        "Recall":    f"{m['rec']:.4f}",
        "Macro-F1":  f"{m['f1']:.4f}",
    })

ablation_df = pd.DataFrame(rows).set_index("Model Variant")
print("\nAblation Study Results (Table 10)")
print(ablation_df.to_string())

ablation_df.to_csv(os.path.join(CFG.output_dir, "ablation_results.csv"))
print(f"\nSaved to {os.path.join(CFG.output_dir, 'ablation_results.csv')}")

In [ ]:
# Visualisation
f1_vals = [ablation_results[n]["f1"] for n, _ in VARIANT_REGISTRY]
names   = [n.split("(")[0].strip() for n, _ in VARIANT_REGISTRY]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, f1_vals, color=["#4C72B0","#55A868","#C44E52","#8172B2","#CCB974"])
ax.set_ylabel("Macro-F1")
ax.set_title("Ablation Study — Multimodal Fusion (Biological Attributes)")
ax.set_ylim(0, 1.0)
for b, v in zip(bars, f1_vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.4f}",
            ha="center", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(CFG.output_dir, "ablation_barplot.png"))
plt.show()

## Interpretation

- **Image features carry the majority of predictive signal** — Image-Only
  performs close to the Full Model.
- **Morphology provides complementary structural information** — the Full
  Model's small gain over Image-Only (∆F1 ≈ 0.002) shows ConvNeXt already
  encodes much shape/compactness/texture information implicitly.
- **Explicit fusion matters** — No-Fusion (simple concatenation) underperforms
  the Full Model, supporting the value of cross-modal Transformer interaction.
- **The Transformer refines cross-modal relationships** — No-Transformer
  (mean-pooling) sits between No-Fusion and the Full Model.
- **Shape alone is insufficient** — Shape-Only collapses to ~0.55 macro-F1,
  confirming that geometric descriptors lack the fine-grained appearance
  information needed for robust protocol inference.